<a href="https://colab.research.google.com/github/aaagairi/Projexa-Pairadox/blob/main/surv_video.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# imports

import os, io, math, time, json, random, zipfile, shutil
from pathlib import Path
from typing import Dict, List, Optional, Tuple
import cv2
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.cuda.amp import GradScaler, autocast
from torch.utils.data import Dataset, DataLoader
import albumentations as A
from albumentations.pytorch import ToTensorV2
from tqdm import tqdm
import torchvision.models as models

In [ ]:
# dataset path

from google.colab import drive
drive.mount('/content/drive')

DATASET_PATH = "/content/drive/MyDrive/Surveillance Project/Surveillance Project/Video"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# config

VIDEO_CONFIG = {
    "num_frames": 12,
    "frame_size": (160,160)
}

LABEL_MAP = {
    "Normal":0,
    "Stealing":1,
    "Shooting":2,
    "Road_accidents":3,
    "Burglary":4,
    "Explosion":5,
    "Assault":6,
    "Arson":7
}

NUM_CLASSES = len(LABEL_MAP)

In [ ]:
# dataset loader

class VideoDataset(Dataset):

    def __init__(self, root_dir):

        self.samples = []

        for class_name in os.listdir(root_dir):

            class_path = os.path.join(root_dir, class_name)

            if class_name.lower() == "normal":

                for video in os.listdir(class_path):

                    self.samples.append(
                        (os.path.join(class_path,video), LABEL_MAP["Normal"])
                    )

            elif class_name.lower() == "suspicious":

                for sub_class in os.listdir(class_path):

                    sub_path = os.path.join(class_path,sub_class)

                    if sub_class not in LABEL_MAP:
                        continue

                    for video in os.listdir(sub_path):

                        self.samples.append(
                            (os.path.join(sub_path,video), LABEL_MAP[sub_class])
                        )

        self.mean = torch.tensor([0.485,0.456,0.406]).view(1,3,1,1)
        self.std = torch.tensor([0.229,0.224,0.225]).view(1,3,1,1)


    def sample_frames(self, video_path):

        cap = cv2.VideoCapture(video_path)

        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

        indices = np.linspace(
            0,
            total_frames-1,
            VIDEO_CONFIG["num_frames"]
        ).astype(int)

        frames = []
        current = 0

        while True:

            ret, frame = cap.read()

            if not ret:
                break

            if current in indices:

                frame = cv2.resize(frame, VIDEO_CONFIG["frame_size"])
                frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

                frames.append(frame)

            current += 1

        cap.release()

        frames = np.array(frames)

        frames = torch.from_numpy(frames).permute(0,3,1,2).float()/255

        frames = (frames - self.mean) / self.std

        return frames


    def __len__(self):
        return len(self.samples)


    def __getitem__(self, idx):

        video_path, label = self.samples[idx]

        video = self.sample_frames(video_path)

        return video, label

In [ ]:
# dataset

dataset = VideoDataset(DATASET_PATH)

print("Total videos:", len(dataset))

Total videos: 133


In [ ]:
# train/validation split

train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size

train_dataset, val_dataset = torch.utils.data.random_split(
    dataset,
    [train_size, val_size]
)

In [ ]:
# dataloaders

train_loader = DataLoader(
    train_dataset,
    batch_size=4,
    shuffle=True,
    num_workers=2
)

val_loader = DataLoader(
    val_dataset,
    batch_size=4,
    shuffle=False
)

In [ ]:
# spatial features extractor (transfer learning)

backbone = models.resnet18(pretrained=True)

backbone.fc = nn.Identity()

/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 165MB/s]


In [ ]:
# Video transformers (cnn for spatial features + transformers for temporal modeling)

class VideoTransformer(nn.Module):

    def __init__(self):

        super().__init__()

        self.backbone = backbone

        self.temporal_transformer = nn.TransformerEncoder(
            nn.TransformerEncoderLayer(
                d_model=512,
                nhead=8
            ),
            num_layers=2
        )

        self.classifier = nn.Linear(512, NUM_CLASSES)


    def forward(self,x):

        B,T,C,H,W = x.shape

        x = x.view(B*T,C,H,W)

        features = self.backbone(x)

        features = features.view(B,T,512)

        features = features.permute(1,0,2)

        temporal_features = self.temporal_transformer(features)

        pooled = temporal_features.mean(dim=0)

        output = self.classifier(pooled)

        return output

In [ ]:
# model

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = VideoTransformer().to(device)

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=3e-4
)
scaler = GradScaler()

/tmp/ipykernel_663/3732055619.py:11: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  self.temporal_transformer = nn.TransformerEncoder(
/tmp/ipykernel_663/2949037595.py:12: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()
/usr/local/lib/python3.12/dist-packages/torch/cuda/amp/grad_scaler.py:31: UserWarning: torch.cuda.amp.GradScaler is enabled, but CUDA is not available.  Disabling.
  super().__init__(


In [ ]:
# training loop

EPOCHS = 5

for epoch in range(EPOCHS):

    model.train()

    total_loss = 0

    for videos, labels in tqdm(train_loader):

        videos = videos.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        with autocast():

            outputs = model(videos)

            loss = criterion(outputs, labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item()

    print("Epoch:",epoch+1,"Loss:",total_loss/len(train_loader))

  0%|          | 0/27 [00:00<?, ?it/s]/tmp/ipykernel_663/3872564513.py:18: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
/usr/local/lib/python3.12/dist-packages/torch/cuda/amp/autocast_mode.py:54: UserWarning: CUDA is not available or torch_xla is imported. Disabling autocast.
  super().__init__(
100%|██████████| 27/27 [04:27<00:00,  9.92s/it]


Epoch: 1 Loss: 2.823779552071183


100%|██████████| 27/27 [04:00<00:00,  8.92s/it]


Epoch: 2 Loss: 1.6589313789650246


100%|██████████| 27/27 [04:10<00:00,  9.28s/it]


Epoch: 3 Loss: 1.030096936005133


100%|██████████| 27/27 [04:16<00:00,  9.51s/it]


Epoch: 4 Loss: 1.0404973576466243


100%|██████████| 27/27 [04:12<00:00,  9.34s/it]

Epoch: 5 Loss: 1.0793394959635205


In [ ]:
# validation

model.eval()

correct = 0
total = 0

with torch.no_grad():

    for videos, labels in val_loader:

        videos = videos.to(device)
        labels = labels.to(device)

        outputs = model(videos)

        _, preds = torch.max(outputs,1)

        correct += (preds == labels).sum().item()
        total += labels.size(0)

print("Validation Accuracy:", correct/total)

Validation Accuracy: 0.4074074074074074
